# Olala mini 7A1B GGUF — llama.cpp fork on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/olala-mini-7a1b-gguf-llamacpp-62dd/colab/Olala-mini-7A1B-GGUF-llamacpp-fork.ipynb)

[`OVHaiLLM/Olala-v1.0-mini-7A1B-GGUF`](https://huggingface.co/OVHaiLLM/Olala-v1.0-mini-7A1B-GGUF) is the Apache-2.0 GGUF of OVHai LLM's (OVHcloud) [`Olala-v1.0-mini-7A1B`](https://huggingface.co/OVHaiLLM/Olala-v1.0-mini-7A1B). The [base card](https://huggingface.co/OVHaiLLM/Olala-v1.0-mini-7A1B) is a **text-only** post-trained chat / tools model: **6.8B total parameters, about 1B active per token**, custom architecture **`olala`**, **32K** context (input + output). The hybrid stack is **36 layers: 29 Mamba-3 and 7 attention** (one in five), each followed by a mixture-of-experts block with **256 routed experts (6 active) and 1 shared expert**. The repo is **not gated**. Licence: **Apache 2.0**.

**Why this model.** It is OVHcloud's first open language-model family ([announcement](https://x.com/OVHcloud/status/2123370331406483592)): a small reasoning model that thinks at four effort levels (`disabled`, `low`, `medium`, `high`) and emits tool calls as XML with JSON arguments. Stock engines cannot load `olala`. The GGUF card says upstream llama.cpp, Ollama, LM Studio, and the Hub "Use this model" apps do **not** support the architecture yet. The Transformers path wants `transformers==5.8.*` plus OVHai's Mamba fork and scattermoe. This notebook stays on the published GGUF and **builds their [llama.cpp fork](https://github.com/OVHai-LLM/llama.cpp)** with CUDA.

**Why a fork compile, not a wheel.** There is no official CUDA tarball for this architecture, and `llama-cpp-python` cannot load `olala`. The GGUF card's "Run with llama.cpp" section is the recipe: clone `OVHai-LLM/llama.cpp`, `cmake` with `-DGGML_CUDA=ON -DLLAMA_USE_PREBUILT_UI=OFF`, then `cmake --build`. This notebook adds **`-DCMAKE_CUDA_ARCHITECTURES=75`** so Colab only compiles Turing (T4 / sm_75), and prints the **wall-clock build time**.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. The free T4 has about **15 GB** of VRAM. Default file: **`Olala-v1.0-mini-7A1B-q8_0-safe2.gguf` (6.9 GB)** from the GGUF card. **`q6_k` (6.3 GB)** then **`q5_k_m` (6.0 GB)** are fallbacks if this GPU OOMs or hits a kernel issue. Do **not** download the 13.6 GB BF16 file on a T4. Demos use **`-c 8192`** (card max 32,768; recurrent models cannot shift a prompt that overruns `-c`). All layers on GPU (`-ngl 99`). Sampler from the cards: **temperature 0.8, top_p 0.9, top_k 40**. Reasoning effort **`low`**, matching the GGUF card's NVIDIA `llama-cli` line (template default is `medium`). Flash attention **on** (`-fa 1`) as the GGUF card states. If sm_75 rejects that kernel, the notebook retries `-fa 0`.

The GGUF chat template file is [`Olala-v1.0-mini-7A1B-chat-template.jinja`](https://huggingface.co/OVHaiLLM/Olala-v1.0-mini-7A1B-GGUF/blob/main/Olala-v1.0-mini-7A1B-chat-template.jinja). Pass `--jinja` **before** `--chat-template-file`.

This walkthrough is **inference only**. It does not train or fine-tune.

**Use cases:** [docs/Olala-mini-7A1B-GGUF-llamacpp-fork.md](https://github.com/donvito/notebooks/blob/melvin/olala-mini-7a1b-gguf-llamacpp-62dd/docs/Olala-mini-7A1B-GGUF-llamacpp-fork.md)

**Links:** [GGUF](https://huggingface.co/OVHaiLLM/Olala-v1.0-mini-7A1B-GGUF) · [base card](https://huggingface.co/OVHaiLLM/Olala-v1.0-mini-7A1B) · [fork](https://github.com/OVHai-LLM/llama.cpp) · [announcement](https://x.com/OVHcloud/status/2123370331406483592)


## 1. Check the GPU

Confirm a T4 (or better) is attached **before** compiling the fork. The next cell prints `torch.cuda.is_available()`, the GPU name, compute capability, and free VRAM, then stops on CPU.


In [ ]:
import subprocess

import torch


def print_vram(label=""):
    # llama.cpp allocates CUDA memory outside PyTorch. nvidia-smi is the measured counter.
    smi = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used,memory.free,memory.total", "--format=csv,noheader"],
        text=True,
    ).strip()
    suffix = f" ({label})" if label else ""
    print(f"nvidia-smi used/free/total{suffix}: {smi}")
    used_mib = int(smi.split(",")[0].strip().split()[0])
    return used_mib


print("torch.cuda.is_available():", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

!nvidia-smi

major, minor = torch.cuda.get_device_capability()
props = torch.cuda.get_device_properties(0)
name = torch.cuda.get_device_name()
total_gb = props.total_memory / 1024**3
free_b, total_b = torch.cuda.mem_get_info()
print()
print(f"GPU name: {name}")
print(f"compute capability: sm_{major}{minor}")
print(f"VRAM total: {total_gb:.1f} GB")
print(f"torch.cuda.mem_get_info free: {free_b / 1024**3:.1f} GB / {total_b / 1024**3:.1f} GB")
print_vram("before build")
print()
print("=== T4-first notes ===")
print("- Architecture is `olala`. Upstream llama.cpp / llama-cpp-python cannot load this GGUF.")
print("- This notebook builds https://github.com/OVHai-LLM/llama.cpp with CUDA for sm_75 only.")
print("- Default file: Olala-v1.0-mini-7A1B-q8_0-safe2.gguf (6.9 GB on the GGUF card).")
print("- Fallbacks if this GPU fails: q6_k (6.3 GB), q5_k_m (6.0 GB), then fewer -ngl.")
print("- Do not download Olala-v1.0-mini-7A1B-bf16.gguf (13.6 GB) on a 15 GB T4.")
print("- Card max context is 32,768. Demos use -c 8192. Recurrent models cannot shift overflow.")
print("- Sampling from the cards: temperature 0.8, top_p 0.9, top_k 40. reasoning_effort=low.")
print("- -fa 1 as the GGUF card. Retry -fa 0 if a Turing kernel errors.")
print("- The fork's older CUDA notes warn quantized files can NaN on GPU; the published GGUF card")
print("  says all four files run on NVIDIA. If you see NaN / ?????, try q6_k then q5_k_m.")
if major < 8:
    print("- This GPU is below sm_80 (no native BF16, no FA2). The path is GGUF + llama.cpp -fa.")
PEAK_VRAM_MIB = print_vram("after gpu check")


## 2. Build the OVHai-LLM llama.cpp fork (CUDA, sm_75)

The GGUF card's CUDA configure line is:

```sh
cmake -B build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release -DLLAMA_USE_PREBUILT_UI=OFF
```

This cell adds `-DCMAKE_CUDA_ARCHITECTURES=75` so nvcc only emits Turing kernels, then `cmake --build build -j$(nproc)` and prints the wall time. `-DLLAMA_USE_PREBUILT_UI=OFF` skips the server web UI download, which the card says does not match this fork yet (the HTTP API is unaffected).

Colab already has a CUDA toolkit. The cell installs `cmake` / `build-essential` if they are missing. Re-running skips the compile when `llama-cli` and `llama-server` are already in `build/bin`.


In [ ]:
%pip install -qqq --no-cache-dir huggingface_hub

import os
import shutil
import subprocess
import time

ROOT = "/content" if os.path.isdir("/content") else "/tmp"
SRC = os.path.join(ROOT, "llama.cpp-olala")
BUILD = os.path.join(SRC, "build")
FORK = "https://github.com/OVHai-LLM/llama.cpp.git"
CLI = os.path.join(BUILD, "bin", "llama-cli")
SERVER = os.path.join(BUILD, "bin", "llama-server")

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
if os.path.isdir("/usr/local/cuda"):
    os.environ["CUDA_HOME"] = "/usr/local/cuda"
    os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ.get("PATH", "")

if shutil.which("cmake") is None or shutil.which("g++") is None:
    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=False,
    )
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "cmake", "build-essential", "git"],
        check=True,
    )

print("cmake:", shutil.which("cmake"))
print("g++:", shutil.which("g++"))
print("nvcc:", shutil.which("nvcc"))
print("nproc:", os.cpu_count())
if shutil.which("nvcc"):
    subprocess.run(["nvcc", "--version"], check=False)

if not os.path.isdir(os.path.join(SRC, ".git")):
    print("cloning", FORK)
    subprocess.run(["git", "clone", "--depth", "1", FORK, SRC], check=True)
else:
    print("reuse clone", SRC)

subprocess.run(["git", "-C", SRC, "log", "-1", "--oneline"], check=False)

already = os.path.isfile(CLI) and os.path.isfile(SERVER)
if already:
    print("already built:", CLI)
    print("already built:", SERVER)
else:
    cmake_cmd = [
        "cmake", "-B", BUILD,
        "-DGGML_CUDA=ON",
        "-DCMAKE_BUILD_TYPE=Release",
        "-DLLAMA_USE_PREBUILT_UI=OFF",
        "-DCMAKE_CUDA_ARCHITECTURES=75",
    ]
    nvcc = shutil.which("nvcc")
    if nvcc:
        cmake_cmd.append(f"-DCMAKE_CUDA_COMPILER={nvcc}")
    print("configure:", " ".join(cmake_cmd))
    subprocess.run(cmake_cmd, cwd=SRC, check=True)

    nproc = str(os.cpu_count() or 2)
    build_cmd = ["cmake", "--build", BUILD, "-j", nproc, "--target", "llama-cli", "llama-server"]
    log_path = os.path.join(ROOT, "olala-llama-build.log")
    print("build:", " ".join(build_cmd))
    print("log:", log_path)
    t0 = time.time()
    with open(log_path, "w") as log_f:
        result = subprocess.run(
            build_cmd,
            cwd=SRC,
            stdout=log_f,
            stderr=subprocess.STDOUT,
        )
    elapsed = time.time() - t0
    print(f"cmake --build wall time: {elapsed:.1f} s ({elapsed / 60:.1f} min)  jobs={nproc}  arch=75")
    if result.returncode != 0:
        print(open(log_path).read()[-4000:])
        raise SystemExit(f"fork build failed with exit {result.returncode}")
    print("build log tail:")
    print("".join(open(log_path).readlines()[-15:]))

assert os.path.isfile(CLI), CLI
assert os.path.isfile(SERVER), SERVER
print("llama-cli:", CLI)
print("llama-server:", SERVER)
print_vram("after build")


## 3. Download the GGUF and the card's chat template

Default file: **`Olala-v1.0-mini-7A1B-q8_0-safe2.gguf`**, **6.9 GB** on the [GGUF files table](https://huggingface.co/OVHaiLLM/Olala-v1.0-mini-7A1B-GGUF). Also download **`Olala-v1.0-mini-7A1B-chat-template.jinja`**. To try a smaller quant on the same T4, set `FILENAME` to `Olala-v1.0-mini-7A1B-q6_k.gguf` (6.3 GB) or `Olala-v1.0-mini-7A1B-q5_k_m.gguf` (6.0 GB).

`HF_HUB_DISABLE_IMPLICIT_TOKEN=1` — the repo is public. Cancel any Colab `HF_TOKEN` prompt.


In [ ]:
import os
import re

from huggingface_hub import hf_hub_download

REPO = "OVHaiLLM/Olala-v1.0-mini-7A1B-GGUF"
FILENAME = "Olala-v1.0-mini-7A1B-q8_0-safe2.gguf"
TEMPLATE_NAME = "Olala-v1.0-mini-7A1B-chat-template.jinja"
FALLBACK_FILES = [
    "Olala-v1.0-mini-7A1B-q8_0-safe2.gguf",  # 6.9 GB
    "Olala-v1.0-mini-7A1B-q6_k.gguf",        # 6.3 GB
    "Olala-v1.0-mini-7A1B-q5_k_m.gguf",      # 6.0 GB
]
N_CTX = 8192
NGL = "99"
FA = "1"
HOST = "127.0.0.1"
PORT = 8088
SERVER_URL = f"http://{HOST}:{PORT}"
SAMPLER = dict(temperature=0.8, top_p=0.9, top_k=40)
REASONING = "low"

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

TEMPLATE = hf_hub_download(REPO, TEMPLATE_NAME)
MODEL = hf_hub_download(REPO, FILENAME)
print("template", TEMPLATE)
print("model", MODEL, "size_gb", round(os.path.getsize(MODEL) / 1024**3, 2))
print_vram("after download (weights still on disk)")

COMMON = [
    "-m", MODEL,
    "-ngl", NGL,
    "-fa", FA,
    "-c", str(N_CTX),
    "--jinja",
    "--chat-template-file", TEMPLATE,
    "--reasoning-effort", REASONING,
    "--temp", str(SAMPLER["temperature"]),
    "--top-p", str(SAMPLER["top_p"]),
    "--top-k", str(SAMPLER["top_k"]),
]


def record_vram(label):
    global PEAK_VRAM_MIB
    used = print_vram(label)
    PEAK_VRAM_MIB = max(PEAK_VRAM_MIB, used)
    return used


def looks_like_kernel_or_nan(text: str) -> bool:
    t = text.lower()
    needles = (
        "cuda_error", "invalid configuration argument", "no kernel image",
        "?????", "out of memory", "failed to allocate",
        "cuda malloc", "illegal memory", "unspecified launch failure",
    )
    if any(n in t for n in needles):
        return True
    return re.search(r"\bnan\b", t) is not None


## 4. Demo 1 — short chat + tokens/s (`llama-cli`)

One user turn, then the process exits (`-st`). The GGUF card's NVIDIA line is `-ngl 99 -fa 1 --temp 0.8 --top-p 0.9 --top-k 40 --reasoning-effort low`. This cell adds `-st -p … -n 192` and the chat-template file. `llama-cli` prints **prompt eval** and **eval** tokens/s at the end (`llama_perf_context_print`). Those are this run's numbers, not the Mac/H100 tables on the cards.

If the first attempt fails with a CUDA kernel / OOM / NaN, the cell retries `-fa 0`, then the next smaller GGUF, then `-ngl 24`.


In [ ]:
import subprocess
import time

CHAT_PROMPT = (
    "In three short sentences, what is a heat pump, and why can it heat a room "
    "while using less electrical energy than a resistive heater?"
)


def run_cli(model, ngl, fa, extra_args):
    cmd = [
        CLI,
        "-m", model,
        "-ngl", str(ngl),
        "-fa", str(fa),
        "-c", str(N_CTX),
        "--jinja",
        "--chat-template-file", TEMPLATE,
        "--reasoning-effort", REASONING,
        "--temp", str(SAMPLER["temperature"]),
        "--top-p", str(SAMPLER["top_p"]),
        "--top-k", str(SAMPLER["top_k"]),
        "-st",
        "-n", "192",
        "-p", CHAT_PROMPT,
        *extra_args,
    ]
    print("exec:", " ".join(cmd[:12]), "...")
    t0 = time.time()
    try:
        proc = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            stdin=subprocess.DEVNULL,
            timeout=300,
        )
        elapsed = time.time() - t0
        return proc.returncode, proc.stdout, elapsed
    except subprocess.TimeoutExpired as exc:
        elapsed = time.time() - t0
        out = exc.stdout or ""
        if isinstance(out, bytes):
            out = out.decode("utf-8", "replace")
        return 124, out + "\n[timeout after 300s]", elapsed


def download_quant(name):
    path = hf_hub_download(REPO, name)
    print("downloaded fallback", name, "size_gb", round(os.path.getsize(path) / 1024**3, 2))
    return path


plans = [
    dict(file=FILENAME, ngl=99, fa=1, note="card default: q8_0-safe2, all layers, -fa 1"),
    dict(file=FILENAME, ngl=99, fa=0, note="same file, -fa 0 (Turing kernel fallback)"),
    dict(file="Olala-v1.0-mini-7A1B-q6_k.gguf", ngl=99, fa=1, note="q6_k 6.3 GB, -fa 1"),
    dict(file="Olala-v1.0-mini-7A1B-q5_k_m.gguf", ngl=99, fa=1, note="q5_k_m 6.0 GB, -fa 1"),
    dict(file="Olala-v1.0-mini-7A1B-q5_k_m.gguf", ngl=24, fa=0, note="q5_k_m, 24 GPU layers, -fa 0"),
]

used_plan = None
cli_out = ""
for plan in plans:
    if plan["file"] != os.path.basename(MODEL):
        MODEL = download_quant(plan["file"])
        FILENAME = plan["file"]
    NGL, FA = str(plan["ngl"]), str(plan["fa"])
    print()
    print("=== try:", plan["note"], "===")
    code, cli_out, elapsed = run_cli(MODEL, plan["ngl"], plan["fa"], [])
    print(cli_out[-6000:] if len(cli_out) > 6000 else cli_out)
    print(f"llama-cli exit {code} in {elapsed:.1f}s")
    record_vram("after llama-cli")
    if code == 0 and not looks_like_kernel_or_nan(cli_out):
        used_plan = plan
        break
    print("that attempt failed; next fallback")
else:
    raise SystemExit(
        "llama-cli did not produce a clean run on this GPU. "
        "See the log above (kernel / OOM / NaN). The 13.6 GB BF16 file does not fit a T4."
    )

print()
print("using", used_plan)
COMMON[1] = MODEL
COMMON[3] = NGL
COMMON[5] = FA
for line in cli_out.splitlines():
    if "tokens per second" in line or "llama_perf" in line:
        print("perf:", line)


## 5. Start `llama-server`

OpenAI-compatible API as on the GGUF card (`reasoning_content`, `content`, `tool_calls`). `--jinja` before `--chat-template-file`. One slot (`-np 1`). Port 8088 to stay clear of Colab's 8080.


In [ ]:
import subprocess
import time

import requests

if "server_proc" in globals() and server_proc is not None and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        server_proc.kill()

LOG = os.path.join(ROOT, "olala-llama-server.log")
log_f = open(LOG, "w")
cmd = [
    SERVER,
    "-m", MODEL,
    "-ngl", NGL,
    "-fa", FA,
    "-c", str(N_CTX),
    "-np", "1",
    "--host", HOST,
    "--port", str(PORT),
    "--jinja",
    "--chat-template-file", TEMPLATE,
    "--reasoning-effort", REASONING,
    "--temp", str(SAMPLER["temperature"]),
    "--top-p", str(SAMPLER["top_p"]),
    "--top-k", str(SAMPLER["top_k"]),
]
print("starting", " ".join(cmd))
server_proc = subprocess.Popen(
    cmd,
    stdout=log_f,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
)

deadline = time.time() + 240
last_err = None
while time.time() < deadline:
    if server_proc.poll() is not None:
        log_f.flush()
        raise SystemExit(
            f"llama-server exited {server_proc.returncode}. Tail of {LOG}:\n"
            + open(LOG).read()[-3000:]
        )
    try:
        health = requests.get(f"{SERVER_URL}/health", timeout=2)
        if health.ok:
            print("llama-server health:", health.text.strip())
            break
        last_err = f"HTTP {health.status_code} {health.text[:200]}"
    except requests.RequestException as exc:
        last_err = str(exc)
    time.sleep(1)
else:
    raise SystemExit(f"llama-server did not become healthy: {last_err}\n" + open(LOG).read()[-3000:])

record_vram("after llama-server load")
print("server", SERVER_URL)


## 6. Demo 2 — tool call / JSON function call

The fork [setup guide](https://github.com/OVHai-LLM/llama.cpp/blob/main/docs/olala-setup.md) and the base card use **`get_weather`** with a `city` argument. Generated calls are XML with JSON arguments in the `tools` channel:

```xml
<toolcalls><call id="call_0"><name>get_weather</name><arguments>{"city": "Paris"}</arguments></call></toolcalls>
```

`llama-server` is supposed to parse that into OpenAI `tool_calls`. The cell prints the raw message, runs a **mocked** weather function, and sends the tool result back for a one-sentence answer. No real weather API.


In [ ]:
import json
import time

WEATHER = {
    "Paris": {"temp_c": 12, "conditions": "light rain", "wind_kmh": 18},
    "Roubaix": {"temp_c": 9, "conditions": "overcast", "wind_kmh": 22},
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "City name"}},
                "required": ["city"],
            },
        },
    }
]


def chat_completions(messages, tools=None, max_tokens=256, reasoning_effort=REASONING):
    payload = {
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": SAMPLER["temperature"],
        "top_p": SAMPLER["top_p"],
        "top_k": SAMPLER["top_k"],
        "reasoning_effort": reasoning_effort,
    }
    if tools:
        payload["tools"] = tools
    t0 = time.time()
    response = requests.post(
        f"{SERVER_URL}/v1/chat/completions",
        json=payload,
        timeout=300,
    )
    elapsed = time.time() - t0
    response.raise_for_status()
    data = response.json()
    choice = data["choices"][0]
    message = choice.get("message") or {}
    usage = data.get("usage") or {}
    prompt_t = usage.get("prompt_tokens") or 0
    completion_t = usage.get("completion_tokens") or 0
    tok_s = (completion_t / elapsed) if elapsed > 0 and completion_t else None
    print(
        f"prompt {prompt_t} | completion {completion_t} | "
        f"finish {choice.get('finish_reason')} | wall {elapsed:.1f}s"
        + (f" | ~{tok_s:.1f} completion tok/s" if tok_s else "")
    )
    record_vram("after chat.completions")
    return data, message


def parse_tool_calls(message):
    calls = message.get("tool_calls") or []
    if calls:
        return calls
    blob = (message.get("content") or "") + (message.get("reasoning_content") or "")
    found = []
    for m in re.finditer(
        r'<call id="([^"]+)"><name>([^<]+)</name><arguments>(.*?)</arguments></call>',
        blob,
        flags=re.S,
    ):
        found.append(
            {
                "id": m.group(1),
                "type": "function",
                "function": {"name": m.group(2), "arguments": m.group(3).strip()},
            }
        )
    return found


messages = [{"role": "user", "content": "What is the weather in Paris?"}]
print("=== user ===")
print(messages[0]["content"])
print()
print("=== first assistant turn (expect a get_weather tool call) ===")
_, message = chat_completions(messages, tools=TOOLS, max_tokens=256, reasoning_effort="low")
print("reasoning_content:", (message.get("reasoning_content") or "")[:800])
print("content:", message.get("content"))
print("tool_calls:", json.dumps(message.get("tool_calls"), indent=2)[:2000])

calls = parse_tool_calls(message)
if not calls:
    raise SystemExit("No tool_calls and no <toolcalls> XML. The model answered in text; see the dump above.")

assistant_msg = {
    "role": "assistant",
    "content": message.get("content") or "",
    "tool_calls": calls,
}
if message.get("reasoning_content"):
    assistant_msg["reasoning_content"] = message["reasoning_content"]
messages.append(assistant_msg)

for call in calls:
    fn = call.get("function") or {}
    name = fn.get("name")
    raw_args = fn.get("arguments") or "{}"
    args = json.loads(raw_args) if isinstance(raw_args, str) else raw_args
    city = (args or {}).get("city", "Paris")
    if name != "get_weather":
        result = {"error": f"unknown tool {name}"}
    else:
        result = WEATHER.get(city, {"temp_c": None, "conditions": "unknown", "city": city, "note": "mocked"})
        result = {"city": city, **result}
    print()
    print("mocked", name, args, "->", result)
    messages.append(
        {
            "role": "tool",
            "tool_call_id": call.get("id") or "call_0",
            "name": name,
            "content": json.dumps(result),
        }
    )

print()
print("=== second turn (tool result in context) ===")
_, final = chat_completions(messages, tools=TOOLS, max_tokens=192, reasoning_effort="low")
print("reasoning_content:", (final.get("reasoning_content") or "")[:500])
print("final:", final.get("content"))


## 7. Demo 3 — few-thousand-token summarisation

The cards give a **32K** window. This cell builds a synthetic operations brief a few thousand tokens long (counted after the request via `usage.prompt_tokens`), asks for an eight-bullet summary, and keeps `-c 8192` so the prompt fits. Prompts longer than `-c` fail: recurrent models cannot shift context.


In [ ]:
def make_brief() -> str:
    header = """NORTHWIND FULFILMENT — WEEKLY OPERATIONS BRIEF
Site: Roubaix DC-3 (mock document for a context-window demo; not real OVHcloud data)
Week of 10–16 November 2026
Audience: site lead, inbound, inventory, outbound, transport, people, safety
Please summarise for a Monday standup. Do not invent sites, vendors, or numbers that are not in this brief.

"""
    streams = [
        (
            "Inbound docks",
            "Dock 1 handled live trailers from Lille and Ghent. Average unload was 41 minutes against a 35-minute target. "
            "Two trailers waited in the yard because the ASN barcodes used last year's vendor prefix. "
            "Dock 2 stayed on the parcel belt. A jam at 14:10 stopped the belt for 18 minutes; maintenance replaced a seized roller. "
            "Dock 3 is still closed for epoxy repair, now due Friday. Overflow used Dock 4 and added about 12 minutes of tug travel per load.",
        ),
        (
            "Inventory accuracy",
            "Cycle counts on aisles R12–R18 finished at 98.4% match to WMS, down from 99.1% last week. "
            "The misses cluster on mixed totes that the putaway RF gun still splits into two licence plates. "
            "A patch is scheduled Tuesday 02:00–04:00. Until then, pickers must scan both plates before leaving the aisle. "
            "Freeze on bulk candy in R04 remains: a temperature logger drifted 1.8 C overnight Thursday.",
        ),
        (
            "Outbound and carrier mix",
            "Orders shipped: 18,440, 4% above plan. Same-day cutoff was pulled forward 20 minutes on Thursday when a tractor failed inspection. "
            "Carrier mix: 61% parcel, 27% groupage, 12% dedicated. Groupage on-time dropped to 91% because the 19:00 trunk to Paris left 35 minutes late. "
            "Customer 'Atlas Kitchen' asked to split pallets by store from Wednesday; that adds one extra label per pallet and about 40 seconds.",
        ),
        (
            "People and safety",
            "Headcount: 212 on payroll, 28 agency. Two agency forklift drivers finished site induction Monday. "
            "Near miss Wednesday: a pallet jack rolled toward the empty-pallet press; the gate interlock held. Action: yellow floor marks and a toolbox talk Friday. "
            "Overtime was 6.2% of hours, mostly outbound Thursday. Flu absences peaked at 9 people on Friday morning.",
        ),
        (
            "Systems",
            "WMS build 2026.11.2 landed Saturday and cleared the duplicate-LPN bug from 3 November. "
            "Label printers 4 and 7 still drop Wi-Fi around 16:00 when the guest network fills; IT will VLAN them this week. "
            "The yard camera on gate B lost focus after rain; a replacement dome arrives Wednesday.",
        ),
        (
            "Next week",
            "Monday: fire-door test at 06:30, expect 12 minutes with no inbound. "
            "Tuesday: WMS putaway patch. Wednesday: Atlas Kitchen store-split labels go live on lane 2 only. "
            "Thursday: peak staffing plan for Black Friday week (draft in the shared folder). "
            "Friday: Dock 3 epoxy inspection. Do not book live unload on Dock 3 until quality signs off.",
        ),
    ]
    parts = [header]
    for i, (title, body) in enumerate(streams, 1):
        extras = "\n".join(
            f"- Checkpoint {j}: {title} remains on the standup list until the owner records a close date."
            for j in range(1, 12)
        )
        follow = (
            f"Owner notes for {title}: keep the same metric definitions as last quarter. "
            f"Do not change the SLA without writing it in this brief. "
            f"If a number is missing, say it is missing. "
        ) * 8
        parts.append(f"## {i}. {title}\n\n{body}\n\n{follow}\n\n{extras}")
    # Extra unique lines so the prompt sits in the few-thousand-token range.
    for n in range(1, 41):
        parts.append(
            f"Appendix line {n:02d}. This paragraph exists so the summarisation prompt is a few thousand "
            f"tokens, not a tweet. It restates that Dock 3 is closed, cycle-count accuracy is 98.4%, "
            f"orders shipped were 18,440, groupage on-time is 91%, overtime is 6.2%, and the WMS patch "
            f"is Tuesday 02:00–04:00. No other sites or figures are introduced here."
        )
    return "\n\n".join(parts)


brief = make_brief()
approx = max(1, len(brief) // 4)
print(f"brief characters: {len(brief):,}  rough token estimate (chars/4): {approx:,}")
print("first 400 chars:\n", brief[:400], "\n...")

sum_messages = [
    {
        "role": "user",
        "content": (
            "Summarise the operations brief below in exactly eight short bullets for a Monday standup. "
            "Cover inbound, inventory, outbound, people/safety, systems, and next week. "
            "Use only numbers that appear in the brief. After the bullets, add one 'watch-outs' line.\n\n"
            + brief
        ),
    }
]
print()
print("=== summarisation ===")
data, summary = chat_completions(sum_messages, tools=None, max_tokens=400, reasoning_effort="low")
usage = data.get("usage") or {}
print("prompt_tokens (server):", usage.get("prompt_tokens"))
print("completion_tokens (server):", usage.get("completion_tokens"))
print()
print(summary.get("content") or "")
if (usage.get("prompt_tokens") or 0) < 1500:
    print("NOTE: prompt_tokens came in under 1,500; the brief generator may need a longer pad on this tokenizer.")


## 8. Peak VRAM, caveats, cleanup

`nvidia-smi` used memory is the peak tracker (`PEAK_VRAM_MIB`). llama.cpp does not allocate through `torch.cuda`.

**Card facts this notebook relies on (not T4 measurements):** 6.8B total / ~1B active; 32K context; Apache-2.0; `olala`; GGUF sizes 13.6 / 6.9 / 6.3 / 6.0 GB; sampling 0.8 / 0.9 / 40; fork required. Mac and H100 tokens/s on the cards are **not** this T4 run.

**T4 risks.** CUDA compile time. sm_75 kernel gaps. Quantized-on-GPU NaN warned in the fork's older setup notes. 6.9 GB weights plus 8K context on 15 GB. Recurrence: do not send a prompt longer than `-c`.

Free the GPU before you open another notebook:


In [ ]:
print(f"peak nvidia-smi used (this session): {PEAK_VRAM_MIB} MiB  ({PEAK_VRAM_MIB / 1024:.2f} GB)")
print("file", FILENAME, "| -ngl", NGL, "| -fa", FA, "| -c", N_CTX)
record_vram("before cleanup")
if "server_proc" in globals() and server_proc is not None and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        server_proc.kill()
        server_proc.wait(timeout=5)
    print("llama-server stopped")
else:
    print("llama-server was not running")
record_vram("after cleanup")
print()
print("Fork:", FORK)
print("GGUF:", f"https://huggingface.co/{REPO}")
print("Template:", TEMPLATE_NAME)
